# Cointegration & Ornstein-Uhlenbeck Demo

Demonstrates stat-arb fundamentals using the C++ engine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

# Optional: import compiled engine
try:
    import stat_arb_mm as mm
    HAS_ENGINE = True
except ImportError:
    HAS_ENGINE = False
    print("C++ engine not compiled. Run: pip install -e .")

## 1. Generate Synthetic Cointegrated Prices

Two assets with:
- Asset A: random walk
- Asset B: follows A with mean-reverting spread

In [ ]:
np.random.seed(42)

# Parameters
n = 5000  # Number of ticks
mu = 0  # Drift
sigma = 0.02  # Volatility
theta = 0.1  # Mean reversion speed
spread_vol = 0.01  # Spread volatility

# Generate Asset A (random walk)
returns_a = np.random.normal(mu, sigma, n)
price_a = 100 * np.exp(np.cumsum(returns_a))

# Generate spread (OU process)
spread = np.zeros(n)
for i in range(1, n):
    spread[i] = spread[i-1] - theta * spread[i-1] + np.random.normal(0, spread_vol)

# Asset B = A + spread
price_b = price_a * np.exp(spread)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
ax1.plot(price_a, label='Asset A', alpha=0.8)
ax1.plot(price_b, label='Asset B', alpha=0.8)
ax1.legend()
ax1.set_title('Cointegrated Price Series')

ax2.plot(spread, color='green')
ax2.axhline(0, color='red', linestyle='--')
ax2.set_title('Mean-Reverting Spread')
plt.tight_layout()
plt.show()

## 2. Estimate Hedge Ratio (OLS)

Find β such that: log(A) - β * log(B) is stationary

In [ ]:
from scipy.stats import linregress

# Log prices
log_a = np.log(price_a)
log_b = np.log(price_b)

# OLS regression
slope, intercept, r_value, p_value, std_err = linregress(log_b, log_a)
beta = slope

print(f"Estimated hedge ratio (β): {beta:.4f}")
print(f"R²: {r_value**2:.4f}")

# Compute spread
spread_est = log_a - beta * log_b

# Z-score
z = (spread_est - spread_est.mean()) / spread_est.std()

plt.figure(figsize=(12, 4))
plt.plot(z)
plt.axhline(2, color='red', linestyle='--', label='Entry ±2σ')
plt.axhline(-2, color='red', linestyle='--')
plt.axhline(0, color='green', linestyle='-', alpha=0.5)
plt.title('Z-Score of Spread')
plt.legend()
plt.show()

## 3. Fit Ornstein-Uhlenbeck Process

dX = θ(μ - X)dt + σdW

Estimate: θ (mean reversion), μ (long-run mean), σ (volatility)

In [ ]:
def fit_ou(spread):
    """Fit OU process to spread via MLE."""
    n = len(spread)
    dt = 1  # 1 tick
    
    # Discrete OU: X_{t+1} = X_t * exp(-θ) + μ(1 - exp(-θ)) + ε
    # Use OLS to estimate parameters
    X = spread[:-1]
    Y = spread[1:]
    
    slope, intercept, r, p, se = linregress(X, Y)
    
    # Extract OU parameters
    theta_hat = -np.log(slope) / dt
    mu_hat = intercept / (1 - slope)
    
    # Residual volatility
    residuals = Y - (slope * X + intercept)
    sigma_hat = np.std(residuals) * np.sqrt(2 * theta_hat / (1 - np.exp(-2 * theta_hat * dt)))
    
    return theta_hat, mu_hat, sigma_hat

theta_hat, mu_hat, sigma_hat = fit_ou(spread_est)

print(f"OU Parameters:")
print(f"  θ (mean reversion): {theta_hat:.4f}")
print(f"  μ (long-run mean):  {mu_hat:.6f}")
print(f"  σ (volatility):     {sigma_hat:.4f}")
print(f"  Half-life: {np.log(2)/theta_hat:.1f} ticks")

## 4. Backtest Simple Stat-Arb Strategy

- Enter when |z| > 2
- Exit when |z| < 0.5

In [ ]:
# Strategy simulation
entry_threshold = 2.0
exit_threshold = 0.5

position = 0  # -1, 0, 1
pnl = 0
entry_price = 0
pnl_curve = []
positions = []

for i in range(len(z)):
    if position == 0:
        if z[i] > entry_threshold:
            # Short spread (sell A, buy B)
            position = -1
            entry_price = spread_est[i]
        elif z[i] < -entry_threshold:
            # Long spread (buy A, sell B)
            position = 1
            entry_price = spread_est[i]
    else:
        if (position == 1 and z[i] > -exit_threshold) or \
           (position == -1 and z[i] < exit_threshold):
            # Exit
            pnl += position * (spread_est[i] - entry_price)
            position = 0
    
    pnl_curve.append(pnl)
    positions.append(position)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(np.array(pnl_curve) * 10000, color='green')  # Scale for visibility
ax1.set_title(f'Cumulative PnL (in bps of spread)')

ax2.plot(positions, color='blue', alpha=0.7)
ax2.set_title('Position')
ax2.set_yticks([-1, 0, 1])
ax2.set_yticklabels(['Short', 'Flat', 'Long'])

plt.tight_layout()
plt.show()

# Metrics
returns = np.diff(pnl_curve)
sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252 * 6.5 * 60)  # Annualized
print(f"\nStrategy Metrics:")
print(f"  Total PnL: {pnl*10000:.2f} bps")
print(f"  Sharpe Ratio: {sharpe:.2f}")
print(f"  Trades: {sum(1 for i in range(1, len(positions)) if positions[i] != positions[i-1])//2}")

## 5. Using the C++ Engine (if compiled)

In [ ]:
if HAS_ENGINE:
    # Create order book
    book = mm.OrderBook()
    ticks = mm.TickNormalizer(0.01)  # $0.01 tick
    
    # Add some orders
    for i in range(10):
        bid_price = ticks.to_ticks(100.0 - i * 0.01)
        ask_price = ticks.to_ticks(100.10 + i * 0.01)
        
        book.add_order(mm.Order(i, i, 0, mm.Side.Buy, mm.Type.Limit, bid_price, 100))
        book.add_order(mm.Order(i+10, i+10, 0, mm.Side.Sell, mm.Type.Limit, ask_price, 100))
    
    print(f"Best Bid: ${ticks.to_price(book.best_bid):.2f}")
    print(f"Best Ask: ${ticks.to_price(book.best_ask):.2f}")
    print(f"Mid Price: ${book.mid_price() * 0.01:.2f}")
    print(f"Spread: {book.spread()} ticks")
    
    # Use SpreadModel
    spread_model = mm.SpreadModel(lookback=100, beta=1.0)
    
    for i in range(200):
        z = spread_model.update(100 + np.random.randn() * 0.1, 
                                 100 + np.random.randn() * 0.1)
    
    print(f"\nSpread Model:")
    print(f"  Z-Score: {spread_model.z_score:.4f}")
    print(f"  Mean: {spread_model.mean:.6f}")
    print(f"  Std: {spread_model.std:.6f}")
else:
    print("Compile C++ engine with: pip install -e .")